# Seizure detection with simulated smart-glasses and around-ear EEG

**Research question:** How much seizure-detection performance is lost when a standard scalp EEG recording is restricted to a small number of physically plausible electrode contacts?

This CHB-MIT starter notebook uses **the same original EDF recording and seizure annotations** for every configuration, eliminating recording-condition differences between configurations. Initial analysis is exploratory. A meaningful performance comparison requires many participants and participant-disjoint validation.

**Important distinction:** EDF *channels* such as `F7-T7` are bipolar differences between two physical electrodes. A four-contact simulated wearable with two bipolar channels must not be described as four independent EEG channels. `F7/T7` and `T7/P7` are **scalp-position proxies**, not actual earbud or glasses recordings.

Source: https://physionet.org/content/chbmit/1.0.0/

## 1. Dependencies and download two example recordings

The download commands pull **two EDFs (approximately 40 MB each)** and one seizure-summary file; you do **not** need the complete 42.6-GB dataset for an initial exploratory plot. `chb01_03.edf` contains one seizure, annotated at 2996–3036 seconds; `chb01_01.edf` contains none.

Download commands are **not** automatically executed. Run the next cell locally to download data. This notebook was created without the raw EEG files, so **no model performance is claimed here**.

In [ ]:
# Install packages once if needed:
# %pip install mne scipy numpy pandas scikit-learn matplotlib seaborn requests

from pathlib import Path
import requests

DATA = Path("data/chbmit/chb01")
DATA.mkdir(parents=True, exist_ok=True)
BASE = "https://physionet.org/files/chbmit/1.0.0/chb01/"
FILES = ["chb01-summary.txt", "chb01_03.edf", "chb01_01.edf"]

def download_file(name):
    dest = DATA / name
    if dest.exists() and dest.stat().st_size > 0:
        print("Present:", dest)
        return dest
    print("Downloading:", name)
    with requests.get(BASE + name, stream=True, timeout=(20, 120)) as r:
        r.raise_for_status()
        with dest.open("wb") as out:
            for chunk in r.iter_content(chunk_size=2**20):
                if chunk:
                    out.write(chunk)
    return dest

# Uncomment next line to download data on your computer:
# paths = [download_file(name) for name in FILES]
print("Run 'paths = [download_file(name) for name in FILES]' to fetch the example files.")

## 2. Read seizure intervals and verify the actual montage

**Always print the actual channel names for each EDF.** CHB-MIT sometimes changes or duplicates channels. Seizure labels are parsed from the subject-specific summary file, not inferred from filename. The same person appears in **chb01 and chb21**; keep those cases grouped when doing patient-disjoint cross-validation.

In [ ]:
import re
import numpy as np
import pandas as pd
import mne

SUMMARY_RE = re.compile(r"File Name:\s*(?P<file>\S+\.edf)(?P<block>.*?)(?=File Name:|\Z)", re.S)
START_RE = re.compile(r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s+seconds", re.I)
END_RE   = re.compile(r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s+seconds", re.I)

def parse_summary(summary_path):
    text = Path(summary_path).read_text(errors="replace")
    result = {}
    for match in SUMMARY_RE.finditer(text):
        starts = list(map(int, START_RE.findall(match.group("block"))))
        ends = list(map(int, END_RE.findall(match.group("block"))))
        if len(starts) != len(ends):
            raise ValueError(f"Mismatched start/end annotations: {match.group('file')}")
        result[match.group("file")] = list(zip(starts, ends))
    return result

summary_path = DATA / "chb01-summary.txt"
if summary_path.exists():
    anns = parse_summary(summary_path)
    print("Example seizure annotations:", anns.get("chb01_03.edf"))
    example = DATA / "chb01_03.edf"
    if example.exists():
        raw = mne.io.read_raw_edf(example, preload=False, verbose="ERROR")
        print("Sampling rate:", raw.info["sfreq"], "Hz")
        print("Recording duration (s):", round(raw.times[-1], 1))
        print("Channels:", raw.ch_names)
else:
    print("Data not downloaded yet; execute the download cell first.")

## 3. Declare physically meaningful electrode layouts

Four-contact reduced montages yield **two bipolar EEG channels**, whereas two-contact single-sided montages yield **one bipolar channel**. The full set uses 18 standard bipolar channels consistently available in common CHB-MIT recordings (verify availability for each case). Near-ear conditions are proxies only: actual behind-ear sensor contacts, references, motion and skin coupling differ from scalp EEG.

- **Glasses / fronto-temporal:** Fp1–F7 and Fp2–F8 (Fp1, F7, Fp2, F8).
- **Temple:** F7–T7 and F8–T8 (F7, T7, F8, T8).
- **Posterior-temporal:** T7–P7 and T8–P8 (T7, P7, T8, P8).
- **Lower temporal:** T7–FT9 and FT10–T8 (T7, FT9, FT10, T8), when available.

The last two are **not** equivalent to four electrodes in an earbud. They only test whether those scalp regions preserve sufficient seizure information.

In [ ]:
MONTAGES = {
    "full_18_bipolar": [
        "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
        "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
        "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
        "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
        "FZ-CZ", "CZ-PZ",
    ],
    "glasses_4_contacts": ["FP1-F7", "FP2-F8"],
    "temple_4_contacts": ["F7-T7", "F8-T8"],
    "posterior_temporal_4_contacts": ["T7-P7", "T8-P8"],
    "lower_temporal_4_contacts": ["T7-FT9", "FT10-T8"],
    "left_temporal_2_contacts": ["F7-T7"],
    "right_temporal_2_contacts": ["F8-T8"],
}
# Optional lower-temporal FT9/FT10 derivations are not present in every recording.
# Keep that arrangement for exploratory comparisons; require only core layouts for CV.
PRIMARY_MONTAGES = {k: v for k, v in MONTAGES.items() if k != "lower_temporal_4_contacts"}

def unique_available_channels(raw, want):
    normalized = {}
    for name in raw.ch_names:
        if name.strip() == "-":
            continue
        # MNE appends -0/-1 to duplicate CHB-MIT channel names (e.g., T8-P8).
        canonical = re.sub(r"-\d+$", "", name.upper().strip())
        normalized.setdefault(canonical, name)
    missing = [name for name in want if name not in normalized]
    return [normalized[name] for name in want if name in normalized], missing

if "raw" in globals():
    for layout, requested in MONTAGES.items():
        found, missing = unique_available_channels(raw, requested)
        print(f"{layout}: {len(found)} bipolar channels; missing={missing}")

## 4. Extract a consistent baseline feature set

For each non-overlapping 10-second window: Welch PSD (delta/theta/alpha/beta/low-gamma), broadband RMS, and normalized line length. A window is positive if it overlaps any annotated seizure. Do not describe this binary **ictal vs non-ictal** task as seizure prediction.

This first-pass approach is designed for interpretability and comparability with the team's PSD + linear-classifier benchmarks. Real monitoring needs artifact rejection, alarm aggregation and prospective validation.

In [ ]:
from scipy.signal import butter, sosfiltfilt, welch

BANDS = {"delta": (1,4), "theta": (4,8), "alpha": (8,13), "beta": (13,30), "low_gamma": (30,40)}
FEATURES_PER_CHANNEL = [f"logpow_{band}" for band in BANDS] + ["log_rms", "log_line_length"]

def window_features(eeg_volts, sfreq):
    """eeg_volts: (n_channels, n_samples). Fixed-size feature vector."""
    sos = butter(4, [0.5, 40], btype="bandpass", fs=sfreq, output="sos")
    filtered = sosfiltfilt(sos, eeg_volts, axis=1)
    freqs, pxx = welch(filtered, fs=sfreq, nperseg=min(int(2*sfreq), filtered.shape[1]), axis=1)
    feats = []
    for ch in range(filtered.shape[0]):
        for lo, hi in BANDS.values():
            mask = (freqs >= lo) & (freqs < hi)
            # SciPy's trapezoid-compatible integration with uniform frequency steps:
            power = np.sum(pxx[ch, mask]) * (freqs[1] - freqs[0])
            feats.append(np.log10(power + 1e-20))
        feats.append(np.log10(np.sqrt(np.mean(filtered[ch]**2)) + 1e-20))
        feats.append(np.log10(np.mean(np.abs(np.diff(filtered[ch]))) + 1e-20))
    return np.asarray(feats)

def has_seizure(start_s, stop_s, seizure_intervals):
    return int(any(start_s < end and stop_s > begin for begin, end in seizure_intervals))

def features_from_edf(edf_path, intervals, montages=MONTAGES, window_s=10):
    raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose="ERROR")
    sfreq = float(raw.info["sfreq"])
    nwin = int(np.floor((raw.n_times / sfreq) / window_s))
    # All layouts are evaluated using EXACTLY the same recording/windows/labels.
    results = {}
    for layout, want in montages.items():
        found, missing = unique_available_channels(raw, want)
        if missing:
            print(f"SKIP {edf_path.name} / {layout}: missing {missing}")
            continue
        picks = mne.pick_channels(raw.ch_names, include=found, ordered=True)
        X, y, time_s = [], [], []
        for win in range(nwin):
            begin = win * window_s
            end = (win + 1) * window_s
            i, j = int(begin * sfreq), int(end * sfreq)
            window = raw.get_data(picks=picks, start=i, stop=j)
            if not np.isfinite(window).all():
                continue
            X.append(window_features(window, sfreq))
            y.append(has_seizure(begin, end, intervals))
            time_s.append(begin)
        feature_names = [f"{ch}:{feature}" for ch in want for feature in FEATURES_PER_CHANNEL]
        data = pd.DataFrame(np.asarray(X), columns=feature_names)
        data["seizure"] = y
        data["start_s"] = time_s
        data["edf_file"] = edf_path.name
        data["case_id"] = edf_path.parent.name
        # chb01 and chb21 share the same actual person
        data["patient_id"] = "chb01_chb21" if edf_path.parent.name in ("chb01", "chb21") else edf_path.parent.name
        results[layout] = data
    raw.close()
    return results

if (DATA / "chb01_03.edf").exists() and summary_path.exists():
    example_feats = features_from_edf(DATA / "chb01_03.edf", anns["chb01_03.edf"])
    for layout, frame in example_feats.items():
        print(layout, "windows", len(frame), "seizure windows", int(frame.seizure.sum()))
else:
    print("Download sample data to run feature extraction.")

## 5. First exploratory result: what changes across montages?

Plot a seizure timeline with the same spectral feature extracted from each configuration. This demonstrates the data workflow, **not** seizure-detection performance. For clinical performance, do not train and test using adjacent windows of the same recording.

In [ ]:
import matplotlib.pyplot as plt

if "example_feats" in globals():
    fig, ax = plt.subplots(figsize=(11,4))
    for layout, df in example_feats.items():
        col = next((c for c in df.columns if c.endswith(":logpow_theta")), None)
        if col:
            # Plot z-scored feature purely for qualitative comparison
            series = df[col].to_numpy()
            z = (series - np.nanmean(series)) / (np.nanstd(series) + 1e-8)
            ax.plot(df.start_s.to_numpy() / 60, z, lw=1, alpha=0.8, label=layout)
    for a,b in anns["chb01_03.edf"]:
        ax.axvspan(a/60, b/60, color="grey", alpha=.2, label="Clinical seizure annotation")
    ax.set(xlabel="Minutes from recording start", ylabel="First-channel theta power (z-score)", title="Exploratory temporal dynamics by montage")
    ax.legend(fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.show()

## 6. Build a patient-disjoint benchmark after downloading more patients

Place any number of subject folders under `data/chbmit/chbXX/`, each with its `chbXX-summary.txt` and EDF files. For valid paired comparisons, only analyze files where **every target configuration is available**. The model uses shrinkage LDA, aligning with the other team members' methods.

Patient-level folds avoid leakage from correlated windows within the same patient. **At least five distinct participants** are required by the implementation below. In a formal paper, use all eligible patients, report event-level sensitivity, false alarms per hour, delay and patient-bootstrap confidence intervals. A window-level F1 score is not enough.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, average_precision_score

ROOT = Path("data/chbmit")

def build_dataset(root=ROOT, required_layouts=PRIMARY_MONTAGES):
    grouped = {name: [] for name in required_layouts}
    for subject_dir in sorted(root.glob("chb*")):
        if not subject_dir.is_dir():
            continue
        spath = subject_dir / f"{subject_dir.name}-summary.txt"
        if not spath.exists():
            continue
        summary = parse_summary(spath)
        for edf in sorted(subject_dir.glob("*.edf")):
            if edf.name not in summary:
                continue
            features = features_from_edf(edf, summary[edf.name], required_layouts)
            if set(features) != set(required_layouts):
                # Paired fairness: skip this EDF for ALL layouts.
                continue
            for layout, frame in features.items():
                grouped[layout].append(frame)
    return {name: pd.concat(parts, ignore_index=True) for name, parts in grouped.items() if parts}

def compare_montages(data):
    if not data:
        raise ValueError("No data found. Download more CHB-MIT subject folders.")
    key = next(iter(data))
    if any(len(df) != len(data[key]) or not df[["edf_file", "start_s", "patient_id"]].equals(data[key][["edf_file", "start_s", "patient_id"]]) for df in data.values()):
        raise ValueError("Layouts do not share exactly the same windows; paired comparison invalid.")
    groups = data[key]["patient_id"].to_numpy()
    if len(np.unique(groups)) < 5:
        raise ValueError("Need at least five distinct participants for five-fold group CV.")
    folds = list(GroupKFold(n_splits=5).split(np.zeros(len(groups)), data[key]["seizure"], groups))
    rows = []
    for name, df in data.items():
        X = df.drop(columns=["seizure", "start_s", "edf_file", "case_id", "patient_id"]).to_numpy()
        y = df["seizure"].to_numpy()
        for fold, (train, test) in enumerate(folds, 1):
            if len(np.unique(y[train])) < 2 or len(np.unique(y[test])) < 2:
                print(f"SKIP {name}, fold {fold}: fewer than two classes")
                continue
            model = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
            model.fit(X[train], y[train])
            prob = model.predict_proba(X[test])[:, 1]
            pred = (prob >= .5).astype(int)  # preliminary fixed threshold only
            rows.append({"layout": name, "fold": fold, "n_test": len(test),
                "seizure_windows": int(y[test].sum()),
                "balanced_accuracy": balanced_accuracy_score(y[test], pred),
                "macro_f1": f1_score(y[test], pred, average="macro", zero_division=0),
                "auprc": average_precision_score(y[test], prob),
                "false_positive_windows_per_hour": (np.sum((y[test] == 0) & (pred == 1)) * 3600 / (len(test)*10)),
            })
    return pd.DataFrame(rows)

# After downloading EDFs and annotations from at least five participants, run:
# all_data = build_dataset()
# cv_results = compare_montages(all_data)
# print(cv_results.groupby("layout")[["balanced_accuracy", "macro_f1", "auprc"]].mean())
print("The group-CV benchmark is defined. Download at least 5 subjects before running it.")

## 7. Planned paper-ready experiments

| Analysis | Comparison | Endpoint |
|---|---|---|
| 1. Channel reduction | Full vs 4 physical contacts vs 2 contacts (if technically available) | Seizure sensitivity, AUPRC, false alarms/hour |
| 2. Placement | Glasses vs temple vs lower/posterior temporal | Paired performance gap; same channel count |
| 3. Random controls | Many random four-contact sets **that form valid bipolar derivations** | Placement percentile and confidence interval |
| 4. Patient generalization | Patient-disjoint CV; chb01 and chb21 grouped | Cross-patient metrics |
| 5. Seizure heterogeneity | Focal temporal vs other locations when labels permit; supplement with SeizeIT2 | Type/location-stratified event sensitivity |
| 6. Real wearables | Apply analogous pipeline to **SeizeIT2 actual behind-ear recordings** | External wearable performance, not same-recording scalp comparison |
| 7. Practicality | Simulated motion artifacts, contact failures, threshold tradeoffs | False alarms/24 h and detection delay |

**Clinical safeguards:** Event-level recall and false-alarm counts require merging adjacent window alarms and scoring each distinct clinical seizure. Choose alarm thresholds in *training/validation only*, then score held-out participants. Ideally assess performance at a matched false-alarm rate, rather than comparing untuned thresholds or accuracy dominated by interictal samples.

**Data-source caveat:** SeizeIT2's public EDFs are behind-ear wearable data (plus ECG/EMG/motion), even though clinical full-scalp vEEG was collected simultaneously to establish labels; the released corpus does not supply a paired full-scalp EEG signal for direct electrode ablation.

Citation: CHB-MIT (Guttag, 2010), SeizeIT2 (Bhagubai et al., Scientific Data, 2025), SzCORE (Dan et al., 2024).